# Factor Model: Step 4c - Combine and Standardize Factors

## Objective
Combine Alpha 101 and Alpha 179 results, standardize, and create final factor loadings.

### Workflow:
1. Load Alpha 101 results from Step 4a
2. Load Alpha 179 results from Step 4b
3. Combine all factors
4. Apply cross-sectional z-score standardization
5. Handle missing values
6. Analyze correlations
7. Save final factor loadings

### Inputs:
- `alpha101_results.parquet` - From Step 4a
- `alpha179_results.parquet` - From Step 4b

### Outputs:
- `russell2000_factor_loadings_step4.parquet` - Final standardized factor loadings
- `factor_loadings_info.csv` - Factor metadata
- `factor_calculation_errors.csv` - Combined error log

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully")

## 1. Load Alpha 101 Results

In [ ]:
print("Loading Alpha 101 results from Step 4a...")

try:
    df_alpha101 = pd.read_parquet('alpha101_results.parquet')
    print(f"✓ Alpha 101 results loaded")
    print(f"  Shape: {df_alpha101.shape}")
    print(f"  Factors: {len(df_alpha101.columns)}")
    
    # Load error log
    try:
        alpha101_errors_df = pd.read_csv('alpha101_errors.csv')
        alpha101_errors = dict(zip(alpha101_errors_df['factor'], alpha101_errors_df['error']))
        print(f"  Errors: {len(alpha101_errors)}")
    except:
        alpha101_errors = {}
        print(f"  No errors found")
        
except FileNotFoundError:
    print("❌ Alpha 101 results not found! Please run Step 4a first.")
    df_alpha101 = pd.DataFrame()
    alpha101_errors = {}

## 2. Load Alpha 179 Results

In [ ]:
print("Loading Alpha 179 results from Step 4b...")

try:
    df_alpha179 = pd.read_parquet('alpha179_results.parquet')
    print(f"✓ Alpha 179 results loaded")
    print(f"  Shape: {df_alpha179.shape}")
    print(f"  Factors: {len(df_alpha179.columns)}")
    
    # Load error log
    try:
        alpha179_errors_df = pd.read_csv('alpha179_errors.csv')
        alpha179_errors = dict(zip(alpha179_errors_df['factor'], alpha179_errors_df['error']))
        print(f"  Errors/Missing: {len(alpha179_errors)}")
    except:
        alpha179_errors = {}
        print(f"  No errors found")
        
except FileNotFoundError:
    print("⚠ Alpha 179 results not found. Continuing with Alpha 101 only.")
    df_alpha179 = pd.DataFrame()
    alpha179_errors = {}

## 3. Combine All Factors

In [ ]:
print("="*80)
print("COMBINING FACTOR LOADINGS")
print("="*80)

# Combine Alpha 101 and Alpha 179
dfs_to_combine = []
if len(df_alpha101) > 0:
    dfs_to_combine.append(df_alpha101)
if len(df_alpha179) > 0:
    dfs_to_combine.append(df_alpha179)

if len(dfs_to_combine) > 0:
    df_factors = pd.concat(dfs_to_combine, axis=1)
    
    print(f"\n✓ Factor loadings combined")
    print(f"  Total shape: {df_factors.shape}")
    print(f"  Rows: {len(df_factors):,}")
    print(f"  Total factors: {len(df_factors.columns)}")
    print(f"    Alpha 101: {len(df_alpha101.columns) if len(df_alpha101) > 0 else 0}")
    print(f"    Alpha 179: {len(df_alpha179.columns) if len(df_alpha179) > 0 else 0}")
    
    print(f"\nFirst few rows:")
    display(df_factors.head(10))
else:
    print("\n❌ No factors to combine!")
    df_factors = pd.DataFrame()

## 4. Apply Cross-Sectional Z-Score Standardization

In [ ]:
print("="*80)
print("CROSS-SECTIONAL Z-SCORE STANDARDIZATION")
print("="*80)

if len(df_factors) > 0:
    print("\nApplying cross-sectional z-score standardization...")
    print("Formula: z = (x - mean) / std (calculated within each date)\n")
    
    def cross_sectional_zscore(group):
        """Standardize within group (date)"""
        return (group - group.mean()) / group.std()
    
    # Apply standardization
    if isinstance(df_factors.index, pd.MultiIndex):
        df_factors_std = df_factors.groupby(level=0).apply(cross_sectional_zscore)
    else:
        df_factors_std = (df_factors - df_factors.mean()) / df_factors.std()
    
    print(f"✓ Standardization complete")
    print(f"  Shape: {df_factors_std.shape}")
    
    print(f"\nStandardized factor statistics:")
    print(df_factors_std.describe())
    
    # Verify standardization
    print(f"\nVerification (should be ~0 mean, ~1 std):")
    print(f"  Mean across all factors: {df_factors_std.mean().mean():.6f}")
    print(f"  Std across all factors: {df_factors_std.std().mean():.6f}")
else:
    print("⚠ No factors to standardize")
    df_factors_std = pd.DataFrame()

## 5. Handle Missing Values

In [ ]:
print("="*80)
print("MISSING VALUE ANALYSIS")
print("="*80)

if len(df_factors_std) > 0:
    # Count missing values
    missing_counts = df_factors_std.isnull().sum()
    missing_pct = (missing_counts / len(df_factors_std)) * 100
    
    missing_summary = pd.DataFrame({
        'Missing_Count': missing_counts,
        'Missing_Pct': missing_pct
    }).sort_values('Missing_Pct', ascending=False)
    
    print(f"\nMissing values summary:")
    print(f"  Total observations: {df_factors_std.size:,}")
    print(f"  Total missing: {df_factors_std.isnull().sum().sum():,}")
    print(f"  Missing %: {(df_factors_std.isnull().sum().sum() / df_factors_std.size) * 100:.2f}%")
    
    print(f"\nFactors with > 10% missing:")
    high_missing = missing_summary[missing_summary['Missing_Pct'] > 10]
    if len(high_missing) > 0:
        display(high_missing.head(20))
    else:
        print("  None")
    
    # Fill missing values
    print(f"\nFilling missing values with 0 (neutral exposure)...")
    df_factors_final = df_factors_std.fillna(0)
    
    print(f"✓ Missing values filled")
    print(f"  Remaining missing: {df_factors_final.isnull().sum().sum()}")
else:
    print("⚠ No factors to analyze")
    df_factors_final = pd.DataFrame()
    missing_summary = pd.DataFrame()

## 6. Factor Correlation Analysis

In [ ]:
print("="*80)
print("FACTOR CORRELATION ANALYSIS")
print("="*80)

if len(df_factors_final) > 0 and len(df_factors_final.columns) > 1:
    print("\nCalculating factor correlation matrix...")
    
    # Sample if too large
    if len(df_factors_final) > 10000:
        sample_size = 10000
        print(f"  Sampling {sample_size:,} rows...")
        corr_matrix = df_factors_final.sample(sample_size).corr()
    else:
        corr_matrix = df_factors_final.corr()
    
    print(f"✓ Correlation matrix calculated")
    print(f"  Shape: {corr_matrix.shape}")
    
    # Summary statistics
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    correlations = upper_triangle.stack()
    
    print(f"\nCorrelation statistics:")
    print(f"  Mean correlation: {correlations.mean():.4f}")
    print(f"  Median correlation: {correlations.median():.4f}")
    print(f"  Max correlation: {correlations.max():.4f}")
    print(f"  Min correlation: {correlations.min():.4f}")
    
    # Highly correlated pairs
    high_corr = correlations[correlations.abs() > 0.8].sort_values(ascending=False)
    
    if len(high_corr) > 0:
        print(f"\n⚠ Highly correlated factor pairs (|corr| > 0.8): {len(high_corr)}")
        print(f"  Top 10:")
        for (f1, f2), corr in high_corr.head(10).items():
            print(f"    {f1} <-> {f2}: {corr:.4f}")
    else:
        print(f"\n✓ No highly correlated factor pairs (|corr| > 0.8)")
    
    # Plot heatmap
    if len(corr_matrix) > 50:
        print(f"\nPlotting correlation heatmap for first 50 factors...")
        corr_subset = corr_matrix.iloc[:50, :50]
    else:
        corr_subset = corr_matrix
    
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(corr_subset, cmap='RdBu_r', center=0, vmin=-1, vmax=1, 
                square=True, ax=ax, cbar_kws={'label': 'Correlation'})
    ax.set_title('Factor Correlation Heatmap (First 50 Factors)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠ Insufficient factors for correlation analysis")
    correlations = pd.Series(dtype=float)
    high_corr = pd.Series(dtype=float)

## 7. Save Final Factor Loadings

In [ ]:
print("="*80)
print("SAVING FACTOR LOADINGS")
print("="*80)

if len(df_factors_final) > 0:
    output_path = 'russell2000_factor_loadings_step4.parquet'
    
    print(f"\nSaving factor loadings to {output_path}...")
    df_factors_final.to_parquet(output_path, compression='snappy')
    
    import os
    file_size_mb = os.path.getsize(output_path) / 1024**2
    
    print(f"✓ Factor loadings saved successfully!")
    print(f"  File: {output_path}")
    print(f"  Size: {file_size_mb:.2f} MB")
    print(f"  Rows: {len(df_factors_final):,}")
    print(f"  Factors: {len(df_factors_final.columns)}")
    
    # Save factor info
    factor_info = pd.DataFrame({
        'factor_name': df_factors_final.columns,
        'source': ['Alpha 101' if 'alpha101' in f else 'Alpha 179' for f in df_factors_final.columns],
        'missing_pct': missing_summary['Missing_Pct'].reindex(df_factors_final.columns).fillna(0).values
    })
    
    factor_info_path = 'factor_loadings_info.csv'
    factor_info.to_csv(factor_info_path, index=False)
    print(f"\n✓ Factor info saved to {factor_info_path}")
    
    # Combined error log
    if alpha101_errors or alpha179_errors:
        error_log = pd.DataFrame([
            {'factor': k, 'error': v, 'source': 'Alpha 101'} for k, v in alpha101_errors.items()
        ] + [
            {'factor': k, 'error': v, 'source': 'Alpha 179'} for k, v in alpha179_errors.items()
        ])
        
        error_log_path = 'factor_calculation_errors.csv'
        error_log.to_csv(error_log_path, index=False)
        print(f"✓ Error log saved to {error_log_path}")
        print(f"  Total errors: {len(error_log)}")
else:
    print("❌ No factors to save!")

## 8. Summary Report

In [ ]:
print("="*80)
print("STEP 4 COMPLETE - SUMMARY REPORT")
print("="*80)

print(f"\nReport Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "="*80)
print("FACTOR CALCULATION RESULTS")
print("="*80)

print(f"\nAlpha 101:")
alpha101_count = len(df_alpha101.columns) if len(df_alpha101) > 0 else 0
print(f"  Target: 101 factors")
print(f"  Calculated: {alpha101_count}")
print(f"  Errors: {len(alpha101_errors)}")
if alpha101_count > 0:
    print(f"  Success rate: {alpha101_count/101*100:.1f}%")

print(f"\nAlpha 179:")
alpha179_count = len(df_alpha179.columns) if len(df_alpha179) > 0 else 0
print(f"  Target: 191 factors (179 implemented)")
print(f"  Calculated: {alpha179_count}")
print(f"  Errors/Missing: {len(alpha179_errors)}")
if alpha179_count > 0:
    print(f"  Success rate: {alpha179_count/191*100:.1f}%")

print(f"\nTotal:")
print(f"  Factors calculated: {alpha101_count + alpha179_count}")
print(f"  Factors in final output: {len(df_factors_final.columns) if len(df_factors_final) > 0 else 0}")

print("\n" + "="*80)
print("DATA QUALITY")
print("="*80)

if len(df_factors_final) > 0:
    print(f"\nFactor loadings shape: {df_factors_final.shape}")
    print(f"  Observations: {len(df_factors_final):,}")
    print(f"  Factors: {len(df_factors_final.columns)}")
    
    print(f"\nStandardization:")
    print(f"  Method: Cross-sectional z-score")
    print(f"  Mean: {df_factors_final.mean().mean():.6f} (should be ~0)")
    print(f"  Std: {df_factors_final.std().mean():.6f} (should be ~1)")
    
    print(f"\nMissing values:")
    print(f"  Total missing (after fill): {df_factors_final.isnull().sum().sum()}")
    if len(missing_summary) > 0:
        print(f"  Factors with >10% missing: {len(missing_summary[missing_summary['Missing_Pct'] > 10])}")
    
    if len(correlations) > 0:
        print(f"\nFactor correlation:")
        print(f"  Mean absolute correlation: {correlations.abs().mean():.4f}")
        print(f"  Highly correlated pairs (|r|>0.8): {len(high_corr)}")

print("\n" + "="*80)
print("OUTPUT FILES")
print("="*80)

if len(df_factors_final) > 0:
    print(f"\nFactor loadings: {output_path}")
    print(f"  Size: {file_size_mb:.2f} MB")
    print(f"  Format: Parquet (standardized, missing filled with 0)")
    
    print(f"\nFactor info: {factor_info_path}")
    print(f"  Contains: factor names, source, missing %")
    
    if alpha101_errors or alpha179_errors:
        print(f"\nError log: {error_log_path}")
        print(f"  Contains: {len(alpha101_errors) + len(alpha179_errors)} failed factors")

print("\n" + "="*80)
print("✅ STEP 4 COMPLETE: FACTOR LOADINGS GENERATED")
print("="*80)
print("\nNext Step: Step 5 - Cross-Sectional Regression")
print("  Regress returns against factor loadings daily to estimate factor returns")